# LLM Free-Form Tagging — Discovering Issue Categories

Rather than classifying fixes into a pre-defined taxonomy, we ask the LLM to assign a
**short free-form tag** to each fix (no choices given).  
We then aggregate the tags to let the natural taxonomy emerge from the data.

## Strategy

1. Call the LLM once per fix, showing `fix_name`, `fix_description`, `reason`.  
2. Ask it to return **only** a concise 2–5 word tag that names the *type* of issue (not the specific content).  
3. Collect tags, normalise (lowercase, strip), and look at frequency distribution.  
4. Embed the tags and cluster them to merge near-synonyms → emergent categories.

**No list of categories is provided to the LLM — the taxonomy is derived entirely from the data.**

In [ ]:
# ── Imports & config ──────────────────────────────────────────────────────────
import os, json, time
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

HERE = Path('.').resolve()
REPO_ROOT = HERE.parent
# RESULTS_PATH = REPO_ROOT / 'results_20260225_163222.csv'
RESULTS_PATH = '../results_20260312_151052_arity.csv'

load_dotenv('../.env')
OPENROUTER_API_KEY= os.getenv('OPENROUTER_API_KEY')

MODEL_NAME = 'anthropic/claude-sonnet-4.6'

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url='https://openrouter.ai/api/v1',
    temperature=0,
)

print(f'Model: {MODEL_NAME}')
print(f'Results file: {RESULTS_PATH}')

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv(RESULTS_PATH)
print(f'Loaded {len(df)} fixes across {df["question_index"].nunique()} questions.')
df.head(2)

## Define the tagging prompt

The LLM is shown the fix and asked to name the *type* of issue in 2–5 words.  
**No category list is provided** — the model must infer the issue type from the fix itself.

In [ ]:
from jinja2 import Environment, FileSystemLoader

jinja_env = Environment(loader=FileSystemLoader("prompts"))
system_template = jinja_env.get_template("system.j2")
user_template = jinja_env.get_template("user.j2")

system_prompt = system_template.render()


def build_user_message(row: pd.Series) -> str:
    return user_template.render(
        fix_name=row['fix_name'],
        fix_description=row['fix_description'],
        reason=row['reason']
    )


def tag_one(row: pd.Series) -> str:
    """Call the LLM and return a free-form issue-type tag."""
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=build_user_message(row)),
    ]
    response = llm.invoke(messages)
    tag = response.content.strip().lower().strip('.')
    return tag


print('Prompt and function defined.')

In [ ]:
# ── Quick smoke-test on 5 rows ─────────────────────────────────────────────────
for i, row in df.head(5).iterrows():
    tag = tag_one(row)
    print(f"[{i}] {row['fix_name'][:60]:60s}  →  {tag}")

In [ ]:
from tqdm import tqdm
# ── Full tagging run ───────────────────────────────────────────────────────────
# ~498 calls.  Adjust SLEEP_BETWEEN_CALLS if you hit rate-limit errors.

SLEEP_BETWEEN_CALLS = 0.2   # seconds
CACHE_PATH = HERE / 'tags_cache.json'

# Load existing cache to resume if interrupted
if CACHE_PATH.exists():
    with open(CACHE_PATH) as f:
        cache: dict = json.load(f)
    print(f'Loaded {len(cache)} cached tags.')
else:
    cache = {}

tags = []
total = len(df)

for i, row in tqdm(df.iterrows(), total=len(df)):
    key = str(i)
    if key in cache:
        tags.append(cache[key])
        continue

    try:
        tag = tag_one(row)
    except Exception as e:
        print(f'  [row {i}] ERROR: {e} — defaulting to "unknown"')
        tag = 'unknown'

    cache[key] = tag
    tags.append(tag)

    if (i + 1) % 50 == 0:
        with open(CACHE_PATH, 'w') as f:
            json.dump(cache, f)
        print(f'  {i + 1}/{total} done\u2026')

    time.sleep(SLEEP_BETWEEN_CALLS)

# Final cache save
with open(CACHE_PATH, 'w') as f:
    json.dump(cache, f)

df['tag'] = tags
print(f'\nTagging complete.  Distinct tags: {df["tag"].nunique()}')

## Explore raw tags

First look: how many distinct tags are there, and how are they distributed?

In [ ]:
tag_counts = df['tag'].value_counts()
print(f'Distinct tags : {len(tag_counts)}')
print(f'Top 30 tags:')
print(tag_counts.head(30).to_string())

In [ ]:
# ── Bar chart of tag frequencies (top 40) ─────────────────────────────────────
top_n = 40
top_tags = tag_counts.head(top_n)

fig, ax = plt.subplots(figsize=(14, 8))
top_tags[::-1].plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title(f'Top {top_n} raw tags assigned by LLM')
ax.set_xlabel('# fixes')
ax.axvline(x=1, color='red', linestyle='--', linewidth=0.8, label='count = 1')
ax.legend()
plt.tight_layout()
plt.savefig('raw_tag_distribution.png', bbox_inches='tight')
plt.show()

# How many tags appear only once?
singletons = (tag_counts == 1).sum()
print(f'Tags appearing exactly once: {singletons} / {len(tag_counts)}')

## Export

Save the annotated dataset: one row per fix, with the raw LLM tag and the tag cluster id.

In [ ]:
out_csv = HERE / 'fixes_tagged_arity.csv'
df[['question_index', 'fix_name', 'fix_description', 'reason', 'tag', 'tag_cluster']].to_csv(out_csv, index=False)
print(f'Saved -> {out_csv}')

# Summary: cluster id, # fixes, top 5 tags
rows = []
for cid in sorted(df['tag_cluster'].unique()):
    fixes_in = df[df['tag_cluster'] == cid]
    top5 = fixes_in['tag'].value_counts().head(5).index.tolist()
    rows.append({'cluster': cid, 'n_fixes': len(fixes_in), 'top_tags': ', '.join(top5)})
summary = pd.DataFrame(rows).sort_values('n_fixes', ascending=False)
summary